In [3]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import os
from glob import glob
import torch
from functorch import vmap

In [4]:
def interpolate(image, coordinates):
    """
    Interpolation function, without a batch size. To make it batched, used vmap from functorch.
    """
    C, H, W = image.shape
    x, y = torch.tensor_split(coordinates, 2, dim=0)
    x = x.view(-1)
    y = y.view(-1)
    x0 = torch.floor(x).long()
    x1 = x0 + 1
    y0 = torch.floor(y).long()
    y1 = y0 + 1

    x0 = torch.clip(x0, 0, W - 1)
    x1 = torch.clip(x1, 0, W - 1)
    y0 = torch.clip(y0, 0, H - 1)
    y1 = torch.clip(y1, 0, H - 1)
    x = torch.clip(x, 0, W - 1)
    y = torch.clip(y, 0, H - 1)

    Ia = image[..., x0, y0]
    Ib = image[..., x0, y1]
    Ic = image[..., x1, y0]
    Id = image[..., x1, y1]

    wa = (x1 - x) * (y1 - y)
    wb = (x1 - x) * (y - y0)
    wc = (x - x0) * (y1 - y)
    wd = (x - x0) * (y - y0)

    _, new_H, new_W = coordinates.shape
    return (wa * Ia + wb * Ib + wc * Ic + wd * Id).view(C, new_H, new_W)

In [5]:
paths = glob("../results/probes_g_hst_mock_psf_deconvolution_slic_*.h5")
hfs = [h5py.File(p, "r") for p in paths]

In [6]:
hfs[0].keys()

<KeysViewHDF5 ['model', 'observation', 'psf', 'reconstruction']>

In [7]:
hfs[0]["reconstruction"]

<HDF5 dataset "reconstruction": shape (30, 1, 64, 64), type "<f4">